In [ ]:



!pip install -q sentence-transformers faiss-cpu numpy


import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import sys
from IPython.display import clear_output

clear_output()
print("✅ All packages installed successfully!\n")


medical_knowledge_base = [
    {"id": 1, "content": "Persistent cough with chest pain may indicate pneumonia, tuberculosis, or lung cancer according to NIH clinical guidelines and recent PubMed studies. Immediate medical consultation and chest X-ray recommended.", "source": "NIH / PubMed"},
    {"id": 2, "content": "Sudden loss of smell and fever is a hallmark symptom of COVID-19 per WHO and CDC guidelines. Other differentials include sinusitis or allergies. PCR testing and isolation advised.", "source": "WHO / CDC"},
    {"id": 3, "content": "Headache accompanied by dizziness may indicate migraine, dehydration, hypertension, or neurological conditions. Evaluation and blood pressure check required.", "source": "UMLS / Medical Literature"},
    {"id": 4, "content": "Sore throat with high fever commonly points to streptococcal pharyngitis or viral influenza. Rapid antigen test and symptomatic care recommended.", "source": "SNOMED CT / PubMed"},
    {"id": 5, "content": "Shortness of breath with fatigue can suggest asthma exacerbation, heart failure, or anemia. Spirometry and echocardiogram recommended if persistent.", "source": "NIH Clinical Guidelines"},
    {"id": 6, "content": "Right-sided abdominal pain may indicate appendicitis or gallbladder issues. Urgent surgical consultation advised.", "source": "WHO Emergency Protocols"},
    {"id": 7, "content": "Skin rash with itching is often allergic reaction, eczema, or contact dermatitis. Antihistamines are commonly used.", "source": "PubMed / NIH"},
    {"id": 8, "content": "Joint pain and swelling typically indicate rheumatoid arthritis or infectious arthritis. Early rheumatology referral recommended.", "source": "UMLS / WHO"},
    {"id": 9, "content": "Lower back pain radiating to legs suggests sciatica or lumbar disc herniation. Conservative management or MRI recommended.", "source": "NIH Spine Guidelines"},
    {"id": 10, "content": "Blurred vision with headache can be sign of hypertensive crisis or glaucoma. Immediate ophthalmology evaluation advised.", "source": "NIH / UMLS"}
]

documents = [entry["content"] for entry in medical_knowledge_base]
sources = [entry["source"] for entry in medical_knowledge_base]




print("🚀 Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("📊 Creating embeddings and FAISS index...")
embeddings = embedding_model.encode(documents, convert_to_numpy=True, normalize_embeddings=True)
embeddings = embeddings.astype('float32')

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"✅ Knowledge base ready! ({len(documents)} evidence-based documents indexed)\n")




def normalize_symptom(text):
    return text.strip().lower()

def llm_interpret_query(user_input):
    return f"Interpreted medical query: Patient reports '{user_input}'. Contextual analysis for possible underlying conditions."

def retrieve_evidence(query, k=3):
    query_emb = embedding_model.encode([query], normalize_embeddings=True).astype('float32')
    distances, indices = index.search(query_emb, k)

    results = []
    for i in range(len(indices[0])):
        idx = indices[0][i]
        if idx < len(documents):
            results.append({
                "content": documents[idx],
                "source": sources[idx],
                "score": float(distances[0][i])
            })
    return results

def generate_response(user_symptom):
    interpreted = llm_interpret_query(user_symptom)
    retrieved = retrieve_evidence(interpreted, k=3)

    print("="*80)
    print("🧠 SMART SYMPTOM-AWARE RAG SYSTEM")
    print("="*80)
    print(f"**User Input:** {user_symptom}")
    print(f"**LLM Interpretation:** {interpreted}\n")
    print("**Evidence-Based Response (RAG Grounded):**")

    if not retrieved:
        print("No relevant medical evidence found.")
    else:
        for i, res in enumerate(retrieved, 1):
            print(f"\n{i}. {res['content']}")
            print(f"   Source: {res['source']}")
            print(f"   Relevance: {res['score']:.4f}")

    print("\n" + "="*80)
    print("⚠️  Disclaimer: This is an AI research prototype for informational purposes only.")
    print("    Not a substitute for professional medical advice. Always consult a doctor.")
    print("="*80)




print("🎯 System Ready! Enter your symptoms below (type 'quit' to stop)\n")

while True:
    try:
        user_input = input("👤 Enter symptoms: ").strip()
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("\nThank you for using the Smart Symptom-Aware System!")
            break
        if user_input:
            generate_response(user_input)
            print("\n")
    except KeyboardInterrupt:
        print("\n\nExited.")
        break

✅ All packages installed successfully!

🚀 Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📊 Creating embeddings and FAISS index...
✅ Knowledge base ready! (10 evidence-based documents indexed)

🎯 System Ready! Enter your symptoms below (type 'quit' to stop)

👤 Enter symptoms: abdoinal pain
🧠 SMART SYMPTOM-AWARE RAG SYSTEM
**User Input:** abdoinal pain
**LLM Interpretation:** Interpreted medical query: Patient reports 'abdoinal pain'. Contextual analysis for possible underlying conditions.

**Evidence-Based Response (RAG Grounded):**

1. Right-sided abdominal pain may indicate appendicitis or gallbladder issues. Urgent surgical consultation advised.
   Source: WHO Emergency Protocols
   Relevance: 0.4829

2. Lower back pain radiating to legs suggests sciatica or lumbar disc herniation. Conservative management or MRI recommended.
   Source: NIH Spine Guidelines
   Relevance: 0.3941

3. Joint pain and swelling typically indicate rheumatoid arthritis or infectious arthritis. Early rheumatology referral recommended.
   Source: UMLS / WHO
   Relevance: 0.3596

⚠️  Disclaimer: Thi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')